# Home Credit Default Risk

## Description of the datasets

#### application_{train|test}.csv
This is the main table, broken into two files for Train (with TARGET) and Test (without TARGET).
Static data for all applications. One row represents one loan in our data sample.

#### bureau.csv
All client's previous credits provided by other financial institutions that were reported to Credit Bureau (for clients who have a loan in our sample).
For every loan in our sample, there are as many rows as number of credits the client had in Credit Bureau before the application date.

#### bureau_balance.csv
Monthly balances of previous credits in Credit Bureau.
This table has one row for each month of history of every previous credit reported to Credit Bureau – i.e the table has (#loans in sample * # of relative previous credits * # of months where we have some history observable for the previous credits) rows.

#### POS_CASH_balance.csv
Monthly balance snapshots of previous POS (point of sales) and cash loans that the applicant had with Home Credit.
This table has one row for each month of history of every previous credit in Home Credit (consumer credit and cash loans) related to loans in our sample – i.e. the table has (#loans in sample * # of relative previous credits * # of months in which we have some history observable for the previous credits) rows.

#### credit_card_balance.csv
Monthly balance snapshots of previous credit cards that the applicant has with Home Credit.
This table has one row for each month of history of every previous credit in Home Credit (consumer credit and cash loans) related to loans in our sample – i.e. the table has (#loans in sample * # of relative previous credit cards * # of months where we have some history observable for the previous credit card) rows.

#### previous_application.csv
All previous applications for Home Credit loans of clients who have loans in our sample.
There is one row for each previous application related to loans in our data sample.

#### installments_payments.csv
Repayment history for the previously disbursed credits in Home Credit related to the loans in our sample.
There is a) one row for every payment that was made plus b) one row each for missed payment.
One row is equivalent to one payment of one installment OR one installment corresponding to one payment of one previous Home Credit credit related to loans in our sample.

#### HomeCredit_columns_description.csv
This file contains descriptions for the columns in the various data files.

## Imports

In [ ]:
import duckdb

# Establish a connection to an in-memory DuckDB instance
con = duckdb.connect(database=':memory:')

# Write the SQL query to inspect the data directly from the CSV
query = """
    SELECT 
        SK_ID_CURR, 
        TARGET, 
        NAME_CONTRACT_TYPE, 
        CODE_GENDER, 
        AMT_INCOME_TOTAL
    FROM read_csv_auto('data/raw/application_train.csv')
    LIMIT 5;
"""

# Execute the query and fetch the result as a pandas DataFrame
df_sample = con.execute(query).df()

# Display the result
display(df_sample)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,AMT_INCOME_TOTAL
0,100002,1,Cash loans,M,202500.0
1,100003,0,Cash loans,F,270000.0
2,100004,0,Revolving loans,M,67500.0
3,100006,0,Cash loans,F,135000.0
4,100007,0,Cash loans,M,121500.0


In [ ]:
# Register files as Views in DuckDB for easier querying later on
con.execute("CREATE VIEW IF NOT EXISTS app_train AS SELECT * FROM read_csv_auto('data/raw/application_train.csv')")
con.execute("CREATE VIEW IF NOT EXISTS app_test AS SELECT * FROM read_csv_auto('data/raw/application_test.csv')")
con.execute("CREATE VIEW IF NOT EXISTS bureau AS SELECT * FROM read_csv_auto('data/raw/bureau.csv')")
con.execute("CREATE VIEW IF NOT EXISTS bureau_balance AS SELECT * FROM read_csv_auto('data/raw/bureau_balance.csv')")
con.execute("CREATE VIEW IF NOT EXISTS credit_card AS SELECT * FROM read_csv_auto('data/raw/credit_card_balance.csv')")
con.execute("CREATE VIEW IF NOT EXISTS pos_cash AS SELECT * FROM read_csv_auto('data/raw/POS_CASH_balance.csv')")
con.execute("CREATE VIEW IF NOT EXISTS install_payments AS SELECT * FROM read_csv_auto('data/raw/installments_payments.csv')")
con.execute("CREATE VIEW IF NOT EXISTS prev_app AS SELECT * FROM read_csv_auto('data/raw/previous_application.csv')")
con.execute("CREATE VIEW IF NOT EXISTS sample_sub AS SELECT * FROM read_csv_auto('data/raw/sample_submission.csv')")

# SQL Queries

### application_train.csv

Exploration

In [ ]:
# See what types of values a column has
con.execute("SELECT DISTINCT NAME_EDUCATION_TYPE FROM app_train;").df()

,NAME_EDUCATION_TYPE
0,Higher education
1,Secondary / secondary special
2,Lower secondary
3,Incomplete higher
4,Academic degree


In [ ]:
# See what types of values a column has and their counts
query_edu = """
SELECT 
    NAME_EDUCATION_TYPE, 
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM app_train), 2) AS percentage
FROM app_train
GROUP BY NAME_EDUCATION_TYPE
ORDER BY count DESC;
"""
display(con.execute(query_edu).df())

,NAME_EDUCATION_TYPE,count,percentage
0,Secondary / secondary special,218391,71.02
1,Higher education,74863,24.34
2,Incomplete higher,10277,3.34
3,Lower secondary,3816,1.24
4,Academic degree,164,0.05


In [ ]:
# Describe the structure of the CSV
schema_query = """
    DESCRIBE SELECT * FROM app_train;
"""

# Fetch and display schema
df_schema = con.execute(schema_query).df()
display(df_schema)

,column_name,column_type,null,key,default,extra
0,SK_ID_CURR,BIGINT,YES,None,None,None
1,TARGET,BIGINT,YES,None,None,None
2,NAME_CONTRACT_TYPE,VARCHAR,YES,None,None,None
3,CODE_GENDER,VARCHAR,YES,None,None,None
4,FLAG_OWN_CAR,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
117,AMT_REQ_CREDIT_BUREAU_DAY,DOUBLE,YES,None,None,None
118,AMT_REQ_CREDIT_BUREAU_WEEK,DOUBLE,YES,None,None,None
119,AMT_REQ_CREDIT_BUREAU_MON,DOUBLE,YES,None,None,None
120,AMT_REQ_CREDIT_BUREAU_QRT,DOUBLE,YES,None,None,None


In [ ]:
# Filter the df by column name
display(df_schema[df_schema['column_name'] == 'AMT_INCOME_TOTAL'])

,column_name,column_type,null,key,default,extra
7,AMT_INCOME_TOTAL,DOUBLE,YES,None,None,None


In [ ]:
# Select only columns of type 'VARCHAR'
df_types = con.execute("DESCRIBE app_train").df()
categorical_cols = df_types[df_types['column_type'] == 'VARCHAR']['column_name'].tolist()

print(f"Categorical columns found: {len(categorical_cols)}")

# See how many distinct values each categorical column has
for col in categorical_cols:
    distinct_count = con.execute(f"SELECT COUNT(DISTINCT {col}) FROM app_train").fetchone()[0]
    print(f"{col}: {distinct_count} categories")

Categorical columns found: 15
NAME_CONTRACT_TYPE: 2 categories
CODE_GENDER: 3 categories
FLAG_OWN_CAR: 2 categories
FLAG_OWN_REALTY: 2 categories
NAME_TYPE_SUITE: 7 categories
NAME_INCOME_TYPE: 8 categories
NAME_EDUCATION_TYPE: 5 categories
NAME_FAMILY_STATUS: 6 categories
NAME_HOUSING_TYPE: 6 categories
OCCUPATION_TYPE: 18 categories
WEEKDAY_APPR_PROCESS_START: 7 categories
ORGANIZATION_TYPE: 58 categories
FONDKAPREMONT_MODE: 4 categories
HOUSETYPE_MODE: 3 categories
WALLSMATERIAL_MODE: 7 categories


In [ ]:
def analyze_categorical(column_name):
    query = f"""
    SELECT 
        {column_name}, 
        COUNT(*) AS number_of_applicants,
        ROUND(AVG(TARGET), 2) AS default_rate,
    FROM app_train
    GROUP BY {column_name}
    ORDER BY default_rate DESC;
    """
    return con.execute(query).df()

# Who is more risky based on education type?
display(analyze_categorical('NAME_EDUCATION_TYPE'))

,NAME_EDUCATION_TYPE,number_of_applicants,default_rate
0,Lower secondary,3816,0.11
1,Secondary / secondary special,218391,0.09
2,Incomplete higher,10277,0.08
3,Higher education,74863,0.05
4,Academic degree,164,0.02


Missing EXT_SOURCE data

In [ ]:
# Count total rows and missing values for specific columns
null_query = """
    SELECT 
        COUNT(*) AS total_applicants,
        COUNT(*) - COUNT(AMT_INCOME_TOTAL) AS missing_income,
        COUNT(*) - COUNT(EXT_SOURCE_1) AS missing_ext_source_1,
        COUNT(*) - COUNT(EXT_SOURCE_2) AS missing_ext_source_2,
        COUNT(*) - COUNT(EXT_SOURCE_3) AS missing_ext_source_3
    FROM app_train;
"""

# Execute and display the result
df_nulls = con.execute(null_query).df()
display(df_nulls)

,total_applicants,missing_income,missing_ext_source_1,missing_ext_source_2,missing_ext_source_3
0,307511,0,173378,660,60965


In [ ]:
# Calculate default rates based on missing data
missingness_query = """
    SELECT
        CASE 
            WHEN EXT_SOURCE_1 IS NULL THEN 'Missing Score'
            ELSE 'Has Score'
        END AS ext_source_1_status,
        COUNT(*) AS total_applicants,
        AVG(TARGET) AS default_rate
    FROM app_train
    GROUP BY 
        ext_source_1_status;
"""

# Execute and display the result
df_missingness = con.execute(missingness_query).df()
display(df_missingness)

,ext_source_1_status,total_applicants,default_rate
0,Has Score,134133,0.074955
1,Missing Score,173378,0.085195


DAYS EMPLOYED

In [ ]:
min_max_query = """
SELECT MIN(DAYS_EMPLOYED), MAX(DAYS_EMPLOYED) 
FROM app_train;
"""

df_min_max = con.execute(min_max_query).df()
display(df_min_max)

,min(DAYS_EMPLOYED),max(DAYS_EMPLOYED)
0,-17912,365243


In [ ]:
income_type_query = """
    SELECT NAME_INCOME_TYPE, COUNT(*)
    FROM app_train
    WHERE DAYS_EMPLOYED = 365243
    GROUP BY NAME_INCOME_TYPE;
    """
df_income_type = con.execute(income_type_query).df()
display(df_income_type)

,NAME_INCOME_TYPE,count_star()
0,Unemployed,22
1,Pensioner,55352


In [ ]:
impact_query = """
SELECT 
    CASE WHEN DAYS_EMPLOYED = 365243 THEN 'Anomalous' ELSE 'Regular' END AS status,
    COUNT(*) AS total_clients,
    AVG(TARGET) AS default_rate
FROM app_train
GROUP BY status;
"""

df_impact = con.execute(impact_query).df()
display(df_impact)

,status,total_clients,default_rate
0,Regular,252137,0.086600
1,Anomalous,55374,0.053996


In [ ]:
# Query to check the cleaning of DAYS_EMPLOYED
query_days_clean = """
    SELECT 
        SK_ID_CURR,
        DAYS_EMPLOYED,
        CASE
            WHEN DAYS_EMPLOYED = 365243 THEN NULL 
            ELSE DAYS_EMPLOYED 
        END AS DAYS_EMPLOYED_CLEAN,
        CASE 
            WHEN DAYS_EMPLOYED = 365243 THEN TRUE 
            ELSE FALSE 
        END AS IS_DAYS_EMPLOYED_ANOM
    FROM app_train
    LIMIT 10;
"""

df_days_check = con.execute(query_days_clean).df()
display(df_days_check)

,SK_ID_CURR,DAYS_EMPLOYED,DAYS_EMPLOYED_CLEAN,IS_DAYS_EMPLOYED_ANOM
0,100002,-637,-637,False
1,100003,-1188,-1188,False
2,100004,-225,-225,False
3,100006,-3039,-3039,False
4,100007,-3038,-3038,False
5,100008,-1588,-1588,False
6,100009,-3130,-3130,False
7,100010,-449,-449,False
8,100011,365243,<NA>,True
9,100012,-2019,-2019,False


In [ ]:
verification_query = """
SELECT 
    NAME_INCOME_TYPE, 
    COUNT(*) AS total,
    AVG(CASE WHEN DAYS_EMPLOYED = 365243 THEN 1 ELSE 0 END) AS ratio_anomali
FROM app_train
GROUP BY NAME_INCOME_TYPE;
"""

df_verification = con.execute(verification_query).df()
display(df_verification)

,NAME_INCOME_TYPE,total,ratio_anomali
0,Pensioner,55362,0.999819
1,State servant,21703,0.000000
2,Unemployed,22,1.000000
3,Commercial associate,71617,0.000000
4,Businessman,10,0.000000
5,Working,158774,0.000000
6,Student,18,0.000000
7,Maternity leave,5,0.000000


OCCUPATION_TYPE

In [ ]:
occupation_query = """
SELECT
    OCCUPATION_TYPE,
    COUNT(*) AS total_applicants
FROM app_train
GROUP BY OCCUPATION_TYPE;
"""

occupation_df = con.execute(occupation_query).df()
display(occupation_df)
    

,OCCUPATION_TYPE,total_applicants
0,Laborers,55186
1,Accountants,9813
2,Managers,21371
3,Cleaning staff,4653
4,Medicine staff,8537
5,Cooking staff,5946
6,High skill tech staff,11380
7,Secretaries,1305
8,NaN,96391
9,Sales staff,32102


In [ ]:
# Replace NaNs with 'Not specified'
query_occ = """
SELECT 
    COALESCE(OCCUPATION_TYPE, 'Not Specified') AS occupation,
    COUNT(*) AS count,
    AVG(TARGET) AS default_rate
FROM app_train
GROUP BY occupation
ORDER BY default_rate DESC;
"""
display(con.execute(query_occ).df())

,occupation,count,default_rate
0,Low-skill Laborers,2093,0.171524
1,Drivers,18603,0.113261
2,Waiters/barmen staff,1348,0.112760
3,Security staff,6721,0.107424
4,Laborers,55186,0.105788
5,Cooking staff,5946,0.104440
6,Sales staff,32102,0.096318
7,Cleaning staff,4653,0.096067
8,Realty agents,751,0.078562
9,Secretaries,1305,0.070498


In [ ]:
# Is there a correlation between income type and missing occupation?
cross_check_query = """
SELECT 
    NAME_INCOME_TYPE,
    COUNT(*) AS total_per_category,
    COUNT(OCCUPATION_TYPE) AS with_occupation,
    COUNT(*) - COUNT(OCCUPATION_TYPE) AS without_occupation,
    ROUND((COUNT(*) - COUNT(OCCUPATION_TYPE)) * 100.0 / COUNT(*), 2) AS percentage_null
FROM app_train
GROUP BY NAME_INCOME_TYPE
ORDER BY without_occupation DESC;
"""

df_cross = con.execute(cross_check_query).df()
display(df_cross)

,NAME_INCOME_TYPE,total_per_category,with_occupation,without_occupation,percentage_null
0,Pensioner,55362,5,55357,99.99
1,Working,158774,133854,24920,15.70
2,Commercial associate,71617,59320,12297,17.17
3,State servant,21703,17916,3787,17.45
4,Unemployed,22,0,22,100.00
5,Student,18,13,5,27.78
6,Businessman,10,8,2,20.00
7,Maternity leave,5,4,1,20.00


Financial Feature Enginnering 

In [ ]:
# Calculating key financial ratios
financial_ratios_query = """
SELECT 
    SK_ID_CURR,
    AMT_INCOME_TOTAL,
    AMT_ANNUITY,
    -- Annuity to Income ratio: how much of the salary goes to the loan?
    AMT_ANNUITY / NULLIF(AMT_INCOME_TOTAL, 0) AS annuity_income_ratio,
    -- Credit to Income ratio: how many years of salary to pay the full debt?
    AMT_CREDIT / NULLIF(AMT_INCOME_TOTAL, 0) AS credit_income_ratio
FROM app_train
LIMIT 10;
"""

df_ratios = con.execute(financial_ratios_query).df()
display(df_ratios)

,SK_ID_CURR,AMT_INCOME_TOTAL,AMT_ANNUITY,annuity_income_ratio,credit_income_ratio
0,100002,202500.0,24700.5,0.121978,2.007889
1,100003,270000.0,35698.5,0.132217,4.790750
2,100004,67500.0,6750.0,0.100000,2.000000
3,100006,135000.0,29686.5,0.219900,2.316167
4,100007,121500.0,21865.5,0.179963,4.222222
5,100008,99000.0,27517.5,0.277955,4.954500
6,100009,171000.0,41301.0,0.241526,9.127053
7,100010,360000.0,42075.0,0.116875,4.250000
8,100011,112500.0,33826.5,0.300680,9.063200
9,100012,135000.0,20250.0,0.150000,3.000000


In [ ]:
# Comparison of financial ratios between Workers and Pensioners
debt_analysis_query = """
SELECT 
    CASE 
        WHEN DAYS_EMPLOYED = 365243 THEN 'Pensioner/Inactive' 
        ELSE 'Working' 
    END AS employment_status,
    COUNT(*) AS total_clients,
    ROUND(AVG(AMT_ANNUITY / NULLIF(AMT_INCOME_TOTAL, 0)), 4) AS avg_annuity_income_ratio,
    ROUND(AVG(AMT_CREDIT / NULLIF(AMT_INCOME_TOTAL, 0)), 4) AS avg_credit_income_ratio,
    ROUND(AVG(TARGET), 4) AS default_rate
FROM app_train
WHERE NAME_INCOME_TYPE IN ('Working', 'Pensioner')
GROUP BY employment_status;
"""

df_debt_analysis = con.execute(debt_analysis_query).df()
display(df_debt_analysis)

,employment_status,total_clients,avg_annuity_income_ratio,avg_credit_income_ratio,default_rate
0,Pensioner/Inactive,55352,0.1975,4.4080,0.0539
1,Working,158784,0.1806,3.8933,0.0959


In [ ]:
# Investigating the 10 "working pensioners" 
investigation_query = """
SELECT 
    SK_ID_CURR,
    NAME_INCOME_TYPE,
    DAYS_EMPLOYED,
    AMT_INCOME_TOTAL,
    DAYS_BIRTH / -365 AS age_years,
    OCCUPATION_TYPE,
    ORGANIZATION_TYPE
FROM app_train
WHERE NAME_INCOME_TYPE = 'Pensioner'
  AND DAYS_EMPLOYED != 365243;
"""

df_anomalous_pensioners = con.execute(investigation_query).df()
display(df_anomalous_pensioners)

,SK_ID_CURR,NAME_INCOME_TYPE,DAYS_EMPLOYED,AMT_INCOME_TOTAL,age_years,OCCUPATION_TYPE,ORGANIZATION_TYPE
0,121224,Pensioner,-1346,94500.0,56.621918,NaN,Business Entity Type 2
1,138428,Pensioner,-5111,76500.0,50.526027,NaN,Industry: type 9
2,223714,Pensioner,-2341,157500.0,60.561644,Laborers,Business Entity Type 3
3,225712,Pensioner,-11194,135000.0,59.271233,NaN,Business Entity Type 3
4,231855,Pensioner,-12865,103500.0,55.539726,Medicine staff,Medicine
5,243990,Pensioner,-1895,67500.0,57.016438,NaN,Business Entity Type 3
6,256117,Pensioner,-8140,78750.0,41.632877,NaN,Military
7,338811,Pensioner,-3261,157500.0,57.512329,NaN,Trade: type 7
8,418551,Pensioner,-1636,148500.0,57.676712,Sales staff,Self-employed
9,437867,Pensioner,-1669,90000.0,50.824658,NaN,School


### bureau.csv

Left Join

In [ ]:
# Aggregating credit history and joining it to the main table
join_query = """
    SELECT 
        app.SK_ID_CURR,
        app.TARGET,
        app.AMT_INCOME_TOTAL,
        COUNT(bur.SK_ID_BUREAU) AS bureau_loan_count,
        COALESCE(SUM(bur.AMT_CREDIT_SUM), 0) AS bureau_total_debt,
        COALESCE(SUM(bur.AMT_CREDIT_SUM_OVERDUE), 0) AS bureau_total_overdue
    FROM app_train AS app
    LEFT JOIN bureau AS bur
        ON app.SK_ID_CURR = bur.SK_ID_CURR
    GROUP BY 
        app.SK_ID_CURR, 
        app.TARGET,
        app.AMT_INCOME_TOTAL
    LIMIT 10;
"""

df_features = con.execute(join_query).df()
display(df_features)

,SK_ID_CURR,TARGET,AMT_INCOME_TOTAL,bureau_loan_count,bureau_total_debt,bureau_total_overdue
0,365733,0,180000.0,1,1.413000e+06,0.0
1,430608,0,450000.0,7,1.101143e+07,0.0
2,360201,0,117000.0,2,5.009985e+05,0.0
3,372026,0,135000.0,7,9.116824e+05,0.0
4,403599,0,112500.0,9,1.507958e+06,0.0
5,187821,0,67500.0,5,5.554368e+05,0.0
6,406120,0,225000.0,4,9.214780e+06,0.0
7,299119,0,270000.0,6,6.017292e+05,0.0
8,411433,0,225000.0,8,1.784830e+06,0.0
9,136306,0,135000.0,5,2.380050e+06,0.0


In [ ]:
query = """
SELECT
    a.SK_ID_CURR,
    COALESCE(b.total_active_debt, 0) AS total_active_debt,
    COALESCE(b.total_closed_debt, 0) AS total_closed_debt,
    COALESCE(b.total_overdue_debt, 0) AS total_overdue_debt,
    COALESCE(b.total_overdue_debt, 0) / NULLIF(COALESCE(b.total_active_debt, 0), 0) AS overdue_to_active_ratio
FROM
    app_train AS a
LEFT JOIN (
    SELECT
        SK_ID_CURR,
        SUM(AMT_CREDIT_SUM) FILTER (WHERE CREDIT_ACTIVE = 'Active') AS total_active_debt,
        SUM(AMT_CREDIT_SUM) FILTER (WHERE CREDIT_ACTIVE = 'Closed') AS total_closed_debt,
        SUM(AMT_CREDIT_SUM_OVERDUE) AS total_overdue_debt
    FROM
        bureau
    GROUP BY
        SK_ID_CURR
) AS b ON a.SK_ID_CURR = b.SK_ID_CURR
LIMIT 10;
"""

# Execute the query and load results into a Pandas DataFrame directly
df_features = con.execute(query).df()

# Display the output directly in the Jupyter Environment
display(df_features)

## Advanced Feature Engineering on bureau.csv

Creiamo la query SQL con `CASE WHEN` per calcolare separatamente il debito attivo (che riduce la capacità di spesa odierna del cliente) e il debito chiuso (indicatore positivo).

In [ ]:
query_bureau_features = """
CREATE OR REPLACE VIEW bureau_agg AS
SELECT 
    SK_ID_CURR,
    COUNT(SK_ID_BUREAU) AS bureau_total_loans,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN COALESCE(AMT_CREDIT_SUM, 0) ELSE 0 END) AS bureau_active_debt,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Closed' THEN COALESCE(AMT_CREDIT_SUM, 0) ELSE 0 END) AS bureau_closed_debt,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN COALESCE(AMT_CREDIT_SUM_DEBT, 0) ELSE 0 END) AS bureau_active_debt_remaining,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN COALESCE(AMT_CREDIT_SUM_OVERDUE, 0) ELSE 0 END) AS bureau_active_debt_overdue
FROM bureau
GROUP BY SK_ID_CURR;
"""
con.execute(query_bureau_features)

# Inspect the new view
display(con.execute("SELECT * FROM bureau_agg LIMIT 5;").df())

## Creazione di Metriche Relative

I valori assoluti significano poco. Creiamo rapporti (ratios) tra il debito pregresso e il reddito attuale.

In [ ]:
query_master = """
CREATE OR REPLACE VIEW master_table AS
SELECT 
    a.*,
    COALESCE(b.bureau_total_loans, 0) AS bureau_total_loans,
    COALESCE(b.bureau_active_debt, 0) AS bureau_active_debt,
    COALESCE(b.bureau_closed_debt, 0) AS bureau_closed_debt,
    COALESCE(b.bureau_active_debt_remaining, 0) AS bureau_active_debt_remaining,
    COALESCE(b.bureau_active_debt_overdue, 0) AS bureau_active_debt_overdue,
    
    -- Metriche Relative
    CASE WHEN a.AMT_INCOME_TOTAL > 0 THEN COALESCE(b.bureau_active_debt, 0) / a.AMT_INCOME_TOTAL ELSE 0 END AS ratio_active_debt_to_income,
    CASE WHEN a.AMT_INCOME_TOTAL > 0 THEN COALESCE(b.bureau_active_debt_overdue, 0) / a.AMT_INCOME_TOTAL ELSE 0 END AS ratio_overdue_debt_to_income
FROM app_train a
LEFT JOIN bureau_agg b ON a.SK_ID_CURR = b.SK_ID_CURR;
"""
con.execute(query_master)

# Esportiamo in un DataFrame Pandas
df_master = con.execute("SELECT * FROM master_table;").df()
print(f"Master table shape: {df_master.shape}")
display(df_master.head())

## Calcolo di WoE e IV in Python

Abbandoniamo SQL temporaneamente. Prendiamo le feature numeriche create per discretizzarle (binning), calcolando il Weight of Evidence (WoE) e l'Information Value (IV), eliminando le variabili che non hanno potere predittivo.

In [ ]:
import pandas as pd
import numpy as np

def calculate_woe_iv(df, feature, target):
    # Binning the feature (numeric) using quantiles
    df['bin'] = pd.qcut(df[feature], q=10, duplicates='drop')
        
    # Calculate events and non-events per bin
    grouped = df.groupby('bin', observed=False).agg(
        events=(target, 'sum'),
        total=(target, 'count')
    )
    grouped['non_events'] = grouped['total'] - grouped['events']
    
    # Avoid division by zero
    grouped['events_pct'] = np.maximum(grouped['events'] / grouped['events'].sum(), 1e-6)
    grouped['non_events_pct'] = np.maximum(grouped['non_events'] / grouped['non_events'].sum(), 1e-6)
    
    # Calculate Weight of Evidence (WoE) and Information Value (IV)
    grouped['WoE'] = np.log(grouped['events_pct'] / grouped['non_events_pct'])
    grouped['IV'] = (grouped['events_pct'] - grouped['non_events_pct']) * grouped['WoE']
    
    iv_total = grouped['IV'].sum()
    
    return grouped, iv_total

# Applichiamo alle feature numeriche create
numerical_features = [
    'AMT_INCOME_TOTAL', 
    'bureau_total_loans',
    'bureau_active_debt', 
    'bureau_closed_debt',
    'bureau_active_debt_remaining',
    'bureau_active_debt_overdue',
    'ratio_active_debt_to_income',
    'ratio_overdue_debt_to_income'
]

iv_results = {}
for feat in numerical_features:
    temp_df = df_master[[feat, 'TARGET']].dropna()
    if temp_df[feat].nunique() > 1:
        try:
            _, iv = calculate_woe_iv(temp_df.copy(), feat, 'TARGET')
            iv_results[feat] = iv
        except Exception:
            iv_results[feat] = 0.0
    else:
        iv_results[feat] = 0.0

# Mostriamo i risultati di IV
iv_df = pd.DataFrame(list(iv_results.items()), columns=['Feature', 'IV']).sort_values(by='IV', ascending=False)
display(iv_df)

# Eliminiamo le feature con Information Value troppo basso (ad esempio, < 0.02)
features_to_drop = iv_df[iv_df['IV'] < 0.02]['Feature'].tolist()
print(f"Features eliminate per basso IV (< 0.02): {features_to_drop}")

# Drop from master dataframe
df_master_filtered = df_master.drop(columns=features_to_drop)
print(f"Nuova shape dopo l'eliminazione: {df_master_filtered.shape}")

## Advanced Feature Engineering on bureau.csv

Creiamo la query SQL con `CASE WHEN` per calcolare separatamente il debito attivo (che riduce la capacità di spesa odierna del cliente) e il debito chiuso (indicatore positivo).

In [ ]:
query_bureau_features = """
CREATE OR REPLACE VIEW bureau_agg AS
SELECT 
    SK_ID_CURR,
    COUNT(SK_ID_BUREAU) AS bureau_total_loans,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN COALESCE(AMT_CREDIT_SUM, 0) ELSE 0 END) AS bureau_active_debt,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Closed' THEN COALESCE(AMT_CREDIT_SUM, 0) ELSE 0 END) AS bureau_closed_debt,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN COALESCE(AMT_CREDIT_SUM_DEBT, 0) ELSE 0 END) AS bureau_active_debt_remaining,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN COALESCE(AMT_CREDIT_SUM_OVERDUE, 0) ELSE 0 END) AS bureau_active_debt_overdue
FROM bureau
GROUP BY SK_ID_CURR;
"""
con.execute(query_bureau_features)

# Inspect the new view
display(con.execute("SELECT * FROM bureau_agg LIMIT 5;").df())

## Creazione di Metriche Relative

I valori assoluti significano poco. Creiamo rapporti (ratios) tra il debito pregresso e il reddito attuale.

In [ ]:
query_master = """
CREATE OR REPLACE VIEW master_table AS
SELECT 
    a.*,
    COALESCE(b.bureau_total_loans, 0) AS bureau_total_loans,
    COALESCE(b.bureau_active_debt, 0) AS bureau_active_debt,
    COALESCE(b.bureau_closed_debt, 0) AS bureau_closed_debt,
    COALESCE(b.bureau_active_debt_remaining, 0) AS bureau_active_debt_remaining,
    COALESCE(b.bureau_active_debt_overdue, 0) AS bureau_active_debt_overdue,
    
    -- Metriche Relative
    CASE WHEN a.AMT_INCOME_TOTAL > 0 THEN COALESCE(b.bureau_active_debt, 0) / a.AMT_INCOME_TOTAL ELSE 0 END AS ratio_active_debt_to_income,
    CASE WHEN a.AMT_INCOME_TOTAL > 0 THEN COALESCE(b.bureau_active_debt_overdue, 0) / a.AMT_INCOME_TOTAL ELSE 0 END AS ratio_overdue_debt_to_income
FROM app_train a
LEFT JOIN bureau_agg b ON a.SK_ID_CURR = b.SK_ID_CURR;
"""
con.execute(query_master)

# Esportiamo in un DataFrame Pandas
df_master = con.execute("SELECT * FROM master_table;").df()
print(f"Master table shape: {df_master.shape}")
display(df_master.head())

## Calcolo di WoE e IV in Python

Abbandoniamo SQL temporaneamente. Prendiamo le feature numeriche create per discretizzarle (binning), calcolando il Weight of Evidence (WoE) e l'Information Value (IV), eliminando le variabili che non hanno potere predittivo.

In [ ]:
import pandas as pd
import numpy as np

def calculate_woe_iv(df, feature, target):
    # Binning the feature (numeric) using quantiles
    df['bin'] = pd.qcut(df[feature], q=10, duplicates='drop')
        
    # Calculate events and non-events per bin
    grouped = df.groupby('bin', observed=False).agg(
        events=(target, 'sum'),
        total=(target, 'count')
    )
    grouped['non_events'] = grouped['total'] - grouped['events']
    
    # Avoid division by zero
    grouped['events_pct'] = np.maximum(grouped['events'] / grouped['events'].sum(), 1e-6)
    grouped['non_events_pct'] = np.maximum(grouped['non_events'] / grouped['non_events'].sum(), 1e-6)
    
    # Calculate Weight of Evidence (WoE) and Information Value (IV)
    grouped['WoE'] = np.log(grouped['events_pct'] / grouped['non_events_pct'])
    grouped['IV'] = (grouped['events_pct'] - grouped['non_events_pct']) * grouped['WoE']
    
    iv_total = grouped['IV'].sum()
    
    return grouped, iv_total

# Applichiamo alle feature numeriche create
numerical_features = [
    'AMT_INCOME_TOTAL', 
    'bureau_total_loans',
    'bureau_active_debt', 
    'bureau_closed_debt',
    'bureau_active_debt_remaining',
    'bureau_active_debt_overdue',
    'ratio_active_debt_to_income',
    'ratio_overdue_debt_to_income'
]

iv_results = {}
for feat in numerical_features:
    temp_df = df_master[[feat, 'TARGET']].dropna()
    if temp_df[feat].nunique() > 1:
        try:
            _, iv = calculate_woe_iv(temp_df.copy(), feat, 'TARGET')
            iv_results[feat] = iv
        except Exception:
            iv_results[feat] = 0.0
    else:
        iv_results[feat] = 0.0

# Mostriamo i risultati di IV
iv_df = pd.DataFrame(list(iv_results.items()), columns=['Feature', 'IV']).sort_values(by='IV', ascending=False)
display(iv_df)

# Eliminiamo le feature con Information Value troppo basso (ad esempio, < 0.02)
features_to_drop = iv_df[iv_df['IV'] < 0.02]['Feature'].tolist()
print(f"Features eliminate per basso IV (< 0.02): {features_to_drop}")

# Drop from master dataframe
df_master_filtered = df_master.drop(columns=features_to_drop)
print(f"Nuova shape dopo l'eliminazione: {df_master_filtered.shape}")

## Advanced Feature Engineering on bureau.csv

Creiamo la query SQL con `CASE WHEN` per calcolare separatamente il debito attivo (che riduce la capacità di spesa odierna del cliente) e il debito chiuso (indicatore positivo).

In [ ]:
query_bureau_features = """
CREATE OR REPLACE VIEW bureau_agg AS
SELECT 
    SK_ID_CURR,
    COUNT(SK_ID_BUREAU) AS bureau_total_loans,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN COALESCE(AMT_CREDIT_SUM, 0) ELSE 0 END) AS bureau_active_debt,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Closed' THEN COALESCE(AMT_CREDIT_SUM, 0) ELSE 0 END) AS bureau_closed_debt,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN COALESCE(AMT_CREDIT_SUM_DEBT, 0) ELSE 0 END) AS bureau_active_debt_remaining,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN COALESCE(AMT_CREDIT_SUM_OVERDUE, 0) ELSE 0 END) AS bureau_active_debt_overdue
FROM bureau
GROUP BY SK_ID_CURR;
"""
con.execute(query_bureau_features)

# Inspect the new view
display(con.execute("SELECT * FROM bureau_agg LIMIT 5;").df())

## Creazione di Metriche Relative

I valori assoluti significano poco. Creiamo rapporti (ratios) tra il debito pregresso e il reddito attuale.

In [ ]:
query_master = """
CREATE OR REPLACE VIEW master_table AS
SELECT 
    a.*,
    COALESCE(b.bureau_total_loans, 0) AS bureau_total_loans,
    COALESCE(b.bureau_active_debt, 0) AS bureau_active_debt,
    COALESCE(b.bureau_closed_debt, 0) AS bureau_closed_debt,
    COALESCE(b.bureau_active_debt_remaining, 0) AS bureau_active_debt_remaining,
    COALESCE(b.bureau_active_debt_overdue, 0) AS bureau_active_debt_overdue,
    
    -- Metriche Relative
    CASE WHEN a.AMT_INCOME_TOTAL > 0 THEN COALESCE(b.bureau_active_debt, 0) / a.AMT_INCOME_TOTAL ELSE 0 END AS ratio_active_debt_to_income,
    CASE WHEN a.AMT_INCOME_TOTAL > 0 THEN COALESCE(b.bureau_active_debt_overdue, 0) / a.AMT_INCOME_TOTAL ELSE 0 END AS ratio_overdue_debt_to_income
FROM app_train a
LEFT JOIN bureau_agg b ON a.SK_ID_CURR = b.SK_ID_CURR;
"""
con.execute(query_master)

# Esportiamo in un DataFrame Pandas
df_master = con.execute("SELECT * FROM master_table;").df()
print(f"Master table shape: {df_master.shape}")
display(df_master.head())

## Calcolo di WoE e IV in Python

Abbandoniamo SQL temporaneamente. Prendiamo le feature numeriche create per discretizzarle (binning), calcolando il Weight of Evidence (WoE) e l'Information Value (IV), eliminando le variabili che non hanno potere predittivo.

In [ ]:
import pandas as pd
import numpy as np

def calculate_woe_iv(df, feature, target):
    # Binning the feature (numeric) using quantiles
    df['bin'] = pd.qcut(df[feature], q=10, duplicates='drop')
        
    # Calculate events and non-events per bin
    grouped = df.groupby('bin', observed=False).agg(
        events=(target, 'sum'),
        total=(target, 'count')
    )
    grouped['non_events'] = grouped['total'] - grouped['events']
    
    # Avoid division by zero
    grouped['events_pct'] = np.maximum(grouped['events'] / grouped['events'].sum(), 1e-6)
    grouped['non_events_pct'] = np.maximum(grouped['non_events'] / grouped['non_events'].sum(), 1e-6)
    
    # Calculate Weight of Evidence (WoE) and Information Value (IV)
    grouped['WoE'] = np.log(grouped['events_pct'] / grouped['non_events_pct'])
    grouped['IV'] = (grouped['events_pct'] - grouped['non_events_pct']) * grouped['WoE']
    
    iv_total = grouped['IV'].sum()
    
    return grouped, iv_total

# Applichiamo alle feature numeriche create
numerical_features = [
    'AMT_INCOME_TOTAL', 
    'bureau_total_loans',
    'bureau_active_debt', 
    'bureau_closed_debt',
    'bureau_active_debt_remaining',
    'bureau_active_debt_overdue',
    'ratio_active_debt_to_income',
    'ratio_overdue_debt_to_income'
]

iv_results = {}
for feat in numerical_features:
    temp_df = df_master[[feat, 'TARGET']].dropna()
    if temp_df[feat].nunique() > 1:
        try:
            _, iv = calculate_woe_iv(temp_df.copy(), feat, 'TARGET')
            iv_results[feat] = iv
        except Exception:
            iv_results[feat] = 0.0
    else:
        iv_results[feat] = 0.0

# Mostriamo i risultati di IV
iv_df = pd.DataFrame(list(iv_results.items()), columns=['Feature', 'IV']).sort_values(by='IV', ascending=False)
display(iv_df)

# Eliminiamo le feature con Information Value troppo basso (ad esempio, < 0.02)
features_to_drop = iv_df[iv_df['IV'] < 0.02]['Feature'].tolist()
print(f"Features eliminate per basso IV (< 0.02): {features_to_drop}")

# Drop from master dataframe
df_master_filtered = df_master.drop(columns=features_to_drop)
print(f"Nuova shape dopo l'eliminazione: {df_master_filtered.shape}")